In [1]:
# ============================================================
# 01_data_preprocessing.ipynb
# KNHANES 2020–2024 Data Preprocessing Pipeline
#
# Paper: A Methodology for Interpretable Health Risk Management:
#        Integrating Counterfactual Explanations and
#        On-Premise Medical LLMs in the Insurance Industry
#
# Data Source: Korea National Health and Nutrition Examination
#              Survey (KNHANES), Korea Disease Control and
#              Prevention Agency (KDCA), 2020–2024
# Note: Raw data files are not included in this repository.
#       Access: https://knhanes.kdca.go.kr
# ============================================================


# ─────────────────────────────────────────────
# Cell 1 | Library Imports
# ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import pyreadstat
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 100)
pd.set_option('display.max_colwidth', None)

print("Libraries loaded successfully.")


# ─────────────────────────────────────────────
# Cell 2 | Variable Definition
# ─────────────────────────────────────────────

# Key variables (identifiers)
key = ['ID', 'year', 'sex']

# Categorical variables
cat_col = [
    'HE_obe',     # Obesity status (19+)
    'BO1_1',      # Weight change in past year
    'BO1_2',      # Weight loss amount in past year
    'BO1_3',      # Weight gain amount in past year
    'BD1_11',     # Alcohol drinking frequency (12+)
    'BD2_1',      # Amount of alcohol per drinking occasion (12+)
    'BS3_1',      # Current cigarette smoking status (adult)
    'BE3_71',     # High-intensity physical activity: work
    'BE3_75',     # High-intensity physical activity: leisure
    'BE3_81',     # Moderate-intensity physical activity: work
    'BE3_91',     # Physical activity: transportation
    'pa_aerobic', # Aerobic physical activity practice rate
    'L_BR_FQ',    # Breakfast frequency per week (past year)
    'BP1',        # Perceived stress level
    'mh_stress',  # Stress recognition rate
    'incm',       # Individual income quartile
    'ho_incm',    # Household income quartile
    'edu',        # Education level (reclassified)
    'BH1',        # Health checkup status (adult)
]

# Continuous variables
num_col = [
    'HE_BMI',   # Body Mass Index (kg/m²)
    'HE_wc',    # Waist circumference (cm)
    'HE_wt',    # Body weight (kg)
    'N_SUGAR',  # Sugar intake (g)
    'N_CHO',    # Carbohydrate intake (g)
    'N_EN',     # Energy intake (kcal)
    'age',      # Age (years)
]

# Target variable
target_col = [
    'HE_DM_HbA1c',  # Diabetes status including HbA1c
                     # Original coding: 1=Normal, 3=Diabetes
]

all_vars = key + cat_col + num_col + target_col

print(f"Total variables selected: {len(all_vars)}")
print(f"  · Key         : {len(key)}")
print(f"  · Categorical : {len(cat_col)}")
print(f"  · Continuous  : {len(num_col)}")
print(f"  · Target      : {len(target_col)}")


# ─────────────────────────────────────────────
# Cell 3 | Load Raw SAS Files (2020–2024)
# ─────────────────────────────────────────────
print("Loading KNHANES raw data files...")

df20, meta20 = pyreadstat.read_sas7bdat('data/hn20_all.sas7bdat')
df21, meta21 = pyreadstat.read_sas7bdat('data/hn21_all.sas7bdat')
df22, meta22 = pyreadstat.read_sas7bdat('data/hn22_all.sas7bdat')
df23, meta23 = pyreadstat.read_sas7bdat('data/hn23_all.sas7bdat')
df24, meta24 = pyreadstat.read_sas7bdat('data/hn24_all.sas7bdat')

print(f"  2020: {df20.shape[0]:,} rows")
print(f"  2021: {df21.shape[0]:,} rows")
print(f"  2022: {df22.shape[0]:,} rows")
print(f"  2023: {df23.shape[0]:,} rows")
print(f"  2024: {df24.shape[0]:,} rows")

# Print variable labels from 2020 metadata (reference year)
print("\n=== Variable Labels (2020 reference) ===")
for name, label in zip(meta20.column_names, meta20.column_labels):
    if name in all_vars:
        print(f"  {name:15s}: {label}")


# ─────────────────────────────────────────────
# Cell 4 | Variable Selection & 5-Year Merge
# ─────────────────────────────────────────────
df20_sel = df20[all_vars].copy()
df21_sel = df21[all_vars].copy()
df22_sel = df22[all_vars].copy()
df23_sel = df23[all_vars].copy()
df24_sel = df24[all_vars].copy()

df_total = pd.concat(
    [df20_sel, df21_sel, df22_sel, df23_sel, df24_sel],
    axis=0
).reset_index(drop=True)

print(f"Merged dataset shape: {df_total.shape}")
print(f"Survey years included: {sorted(df_total['year'].unique())}")


# ─────────────────────────────────────────────
# Cell 5 | Target Variable Encoding
# ─────────────────────────────────────────────
# Original: 1 = Normal, 3 = Diabetes
# Recoded : 0 = Non-diabetic, 1 = Diabetic
# Values outside {1, 3} (e.g., borderline, missing) → excluded

df_total['HE_DM_HbA1c'] = df_total['HE_DM_HbA1c'].map({1: 0, 3: 1})

print("Target variable distribution after encoding:")
print(df_total['HE_DM_HbA1c'].value_counts(dropna=False))
print(f"\n  0 (Non-diabetic) : {(df_total['HE_DM_HbA1c'] == 0).sum():,}")
print(f"  1 (Diabetic)     : {(df_total['HE_DM_HbA1c'] == 1).sum():,}")
print(f"  NaN (excluded)   : {df_total['HE_DM_HbA1c'].isna().sum():,}")


# ─────────────────────────────────────────────
# Cell 6 | Filter Valid Target Rows & Adult (19+)
# ─────────────────────────────────────────────
# Step 1: Keep only valid target values (0 or 1)
df_total = df_total[df_total['HE_DM_HbA1c'].isin([0, 1])].reset_index(drop=True)

# Step 2: Filter adults aged 19 and above
df_total = df_total[df_total['age'] >= 19].reset_index(drop=True)

print(f"After valid target filter + adult filter: {df_total.shape[0]:,} rows")


# ─────────────────────────────────────────────
# Cell 7 | Missing Value Handling
# ─────────────────────────────────────────────
print("=== Missing Value Report (before removal) ===")
missing = df_total[all_vars].isnull().sum()
missing_pct = (missing / len(df_total) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing (%)': missing_pct
}).query('`Missing Count` > 0').sort_values('Missing (%)', ascending=False)
print(missing_df)

# Complete case analysis: drop rows with any missing value
df_total = df_total.dropna(subset=all_vars).reset_index(drop=True)

print(f"\nAfter complete case analysis: {df_total.shape[0]:,} rows")


# ─────────────────────────────────────────────
# Cell 8 | Data Type Casting
# ─────────────────────────────────────────────
# Target → integer
df_total[target_col] = df_total[target_col].astype(int)

# Categorical → integer
for col in cat_col:
    df_total[col] = df_total[col].astype(int)

# Continuous → float
for col in num_col:
    df_total[col] = df_total[col].astype(float)

print("Data types after casting:")
print(df_total[all_vars].dtypes)


# ─────────────────────────────────────────────
# Cell 9 | Outlier Removal (Continuous Variables)
# ─────────────────────────────────────────────
# Rule: remove values outside mean ± 3SD per continuous variable

print("=== Outlier Removal (Mean ± 3SD) ===")
before = len(df_total)

for col in num_col:
    mean = df_total[col].mean()
    std  = df_total[col].std()
    lower, upper = mean - 3 * std, mean + 3 * std
    removed = ((df_total[col] < lower) | (df_total[col] > upper)).sum()
    df_total = df_total[
        (df_total[col] >= lower) & (df_total[col] <= upper)
    ].reset_index(drop=True)
    print(f"  {col:12s} | range [{lower:.2f}, {upper:.2f}] | removed {removed} rows")

print(f"\nRows before: {before:,} → after: {len(df_total):,} "
      f"(removed {before - len(df_total):,})")


# ─────────────────────────────────────────────
# Cell 10 | Gender Split
# ─────────────────────────────────────────────
# sex: 1 = Male, 2 = Female

df_male   = df_total[df_total['sex'] == 1].reset_index(drop=True)
df_female = df_total[df_total['sex'] == 2].reset_index(drop=True)

print("=== Final Dataset Summary ===")
print(f"  Total  : {len(df_total):,}")
print(f"  Male   : {len(df_male):,}  "
      f"(Diabetic: {df_male['HE_DM_HbA1c'].sum():,}, "
      f"{df_male['HE_DM_HbA1c'].mean()*100:.1f}%)")
print(f"  Female : {len(df_female):,}  "
      f"(Diabetic: {df_female['HE_DM_HbA1c'].sum():,}, "
      f"{df_female['HE_DM_HbA1c'].mean()*100:.1f}%)")


# ─────────────────────────────────────────────
# Cell 11 | Save Preprocessed Data
# ─────────────────────────────────────────────
df_total.to_csv('outputs/df_total.csv',  index=False)
df_male.to_csv('outputs/df_male.csv',    index=False)
df_female.to_csv('outputs/df_female.csv', index=False)

print("Preprocessed files saved to outputs/")
print("  · df_total.csv")
print("  · df_male.csv")
print("  · df_female.csv")

Libraries loaded successfully.
Total variables selected: 30
  · Key         : 3
  · Categorical : 19
  · Continuous  : 7
  · Target      : 1
Loading KNHANES raw data files...
  2020: 7,359 rows
  2021: 7,090 rows
  2022: 6,265 rows
  2023: 6,929 rows
  2024: 6,997 rows

=== Variable Labels (2020 reference) ===
  ID             : 개인아이디
  year           : 조사연도
  sex            : 성별
  age            : 만나이
  incm           : 소득4분위수(개인)
  ho_incm        : 소득4분위수(가구)
  edu            : 교육수준 재분류 코드
  BH1            : (성인) 건강검진 수진 여부
  BO1_1          : (성인) 1년간 체중 변화 여부
  BO1_2          : (성인) 1년간 체중 감소량
  BO1_3          : (성인) 1년간 체중 증가량
  BD1_11         : (만12세이상) 1년간 음주빈도
  BD2_1          : (만12세이상) 한번에 마시는 음주량
  BP1            : 평소 스트레스 인지 정도
  mh_stress      : 스트레스 인지율
  BS3_1          : (성인) 현재 일반담배(궐련) 흡연 여부
  BE3_71         : 고강도 신체활동 여부: 일
  BE3_81         : 중강도 신체활동 여부: 일
  BE3_91         : 신체활동 여부: 장소이동
  BE3_75         : 고강도 신체활동 여부: 여가
  pa_aerobic     : 유산소 신체활동 실천율
  HE_wt      